In [6]:
# # 이름 중복 방지 (bad 이미지와 good 이미지에 서로 동일한 이름이 존재해서 train, test, val split 과정애서
#                   시스템이 혼동하는 과정을 막고자 이름을 재정의함)
# 예시: bad 이미지 - 001_2020...bpm 형식 / good 이미지 - 001.jpg 형식. 둘이 001로 동일해서 시스템이 앞서
#       bad 이미지 001을 잡아낸 후, good 이미지의 001을 잡아버리면 뒤에걸로 앞의 것을 덮어씌우는 현상이 발생해버림


# import os

# PROJECT_DIR = '/content/drive/MyDrive/YOLO_Project'
# GOOD_DIR = os.path.join(PROJECT_DIR, 'dataset', 'good')

# if os.path.exists(GOOD_DIR):
#     # 1. good 폴더 내 모든 파일 탐색
#     files = os.listdir(GOOD_DIR)

#     # 1-1. 이미지 파일명에 'good_' 접두사 추가 (이름 중복 방지)
#     renamed_count = 0
#     for filename in files:
#         if filename.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
#             if not filename.startswith('good_'):
#                 old_path = os.path.join(GOOD_DIR, filename)
#                 new_filename = f"good_{filename}"
#                 new_path = os.path.join(GOOD_DIR, new_filename)
#                 os.rename(old_path, new_path)
#                 renamed_count += 1

#     print(f"✅ {renamed_count}개 good 이미지 파일명 변경 완료 ('good_' 접두사 추가)")

#     # 2. 변경된 이미지들에 대응하는 빈 .txt 라벨 파일 생성
#     updated_files = os.listdir(GOOD_DIR)
#     txt_count = 0

#     for filename in updated_files:
#         if filename.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
#             base_name = os.path.splitext(filename)[0]
#             txt_path = os.path.join(GOOD_DIR, f"{base_name}.txt")

#             # txt 파일이 없을 때만 빈 파일 생성
#             if not os.path.exists(txt_path):
#                 open(txt_path, 'w').close() # 0 Byte 빈 파일 작성
#                 txt_count += 1

#     print(f"🎉 {txt_count}개 good 이미지에 대한 빈 라벨(.txt) 파일 생성 완료!")

# else:
#     print(f"⚠️ '{GOOD_DIR}' 경로를 찾을 수 없습니다. 경로를 다시 확인해 주세요.")


# 한 번만 실행하면 돼서 셀 코드 1번 실행 후, 오류 방지를 위해 코드 all 주석 처리

✅ 500개 good 이미지 파일명 변경 완료 ('good_' 접두사 추가)
🎉 500개 good 이미지에 대한 빈 라벨(.txt) 파일 생성 완료!


##### 위 셀의 과정은 good 이미지들의 이름을 재정의하고 그에 상응하는 .txt 파일을 생성 (good은 정상품이므로 txt가 모두 비어있다)  
##### bad 이미지는 이미 상응하는 txt 파일들이 있으므로 동일한 루트상에 txt 파일들 드라이브에 옮김  
##### good과 bad 모두 상응하는 .txt(labels) 파일을 만들었다면 이 상태로 train, val, test split 진행


In [2]:
# 구글 드라이브 마운트 및 환경 설정

import os
import shutil
import random
import urllib.request
import yaml
from google.colab import drive
from ultralytics import YOLO

# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

# 2. 프로젝트 최상위 경로 설정
PROJECT_DIR = '/content/drive/MyDrive/YOLO_Project'
RAW_DATA_DIR = os.path.join(PROJECT_DIR, 'dataset')     # 원본 bad, good 폴더 위치
DET_DATA_DIR = os.path.join(PROJECT_DIR, 'dataset_det') # Detection 전용 분할 폴더

os.makedirs(PROJECT_DIR, exist_ok=True)
%cd {PROJECT_DIR}
print(f"현재 작업 디렉터리: {os.getcwd()}")

# 3. 필수 라이브러리 설치
!pip install -q ultralytics huggingface_hub

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/YOLO_Project
현재 작업 디렉터리: /content/drive/MyDrive/YOLO_Project


In [4]:
# yolov10 가중치 수동 다운로드 (다운로드 오류 방지)

v10_weight_path = os.path.join(PROJECT_DIR, "yolov10n.pt")
v10_weight_url = "https://github.com/THU-MIG/yolov10/releases/download/v1.0/yolov10n.pt"

if not os.path.exists(v10_weight_path):
    print("YOLOv10 가중치 다운로드 중...")
    urllib.request.urlretrieve(v10_weight_url, v10_weight_path)
    print("YOLOv10 가중치 다운로드 완료!")
else:
    print("이미 YOLOv10 가중치가 존재합니다.")

이미 YOLOv10 가중치가 존재합니다.


In [7]:
# train/val/test 하위 디렉터리 생성
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(DET_DATA_DIR, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(DET_DATA_DIR, split, 'labels'), exist_ok=True)

# bad, good 폴더에서 이미지와 txt 라벨 쌍 매칭
all_pairs = []
for category in ['bad', 'good']:
    cat_dir = os.path.join(RAW_DATA_DIR, category)
    if not os.path.exists(cat_dir):
        print(f"⚠️ 경고: '{cat_dir}' 경로를 찾을 수 없습니다.")
        continue

    for file in os.listdir(cat_dir):
        if file.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            base_name = os.path.splitext(file)[0]
            img_path = os.path.join(cat_dir, file)
            txt_path = os.path.join(cat_dir, base_name + '.txt')
            all_pairs.append((img_path, txt_path, file, base_name + '.txt'))

random.seed(42)
random.shuffle(all_pairs)

# 8:1:1 비율 분할
n_total = len(all_pairs)
n_train = int(n_total * 0.8)
n_val = int(n_total * 0.1)

splits = {
    'train': all_pairs[:n_train],
    'val': all_pairs[n_train:n_train + n_val],
    'test': all_pairs[n_train + n_val:]
}

# 파일 복사 진행
for split_name, pairs in splits.items():
    for img_src, txt_src, img_name, txt_name in pairs:
        # 이미지 복사
        shutil.copy(img_src, os.path.join(DET_DATA_DIR, split_name, 'images', img_name))
        # 라벨 파일 복사 (good 빈 txt 파일 포함)
        if os.path.exists(txt_src):
            shutil.copy(txt_src, os.path.join(DET_DATA_DIR, split_name, 'labels', txt_name))

print(f"🎉 Detection 데이터셋 분할 완료! 총 {n_total}장 (Train: {n_train}, Val: {n_val}, Test: {n_total - n_train - n_val})")

🎉 Detection 데이터셋 분할 완료! 총 1000장 (Train: 800, Val: 100, Test: 100)


In [8]:
# data.yaml 설정 파일 생성

yaml_content = {
    'path': DET_DATA_DIR,
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': 2,
    'names': ['bad', 'good']  # 라벨 클래스 순서 및 이름
}

yaml_path = os.path.join(DET_DATA_DIR, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"🎉 data.yaml 생성 완료: {yaml_path}")

🎉 data.yaml 생성 완료: /content/drive/MyDrive/YOLO_Project/dataset_det/data.yaml


In [16]:
# # yolov10과 yolov12를 실행하던 도중, v12 에서 오류가 발생해 다시 순차적으로 실행하기 위해
# # yolov10의 runs_det 폴더 깔끔하게 초기화

# import os
# import shutil

# PROJECT_DIR = '/content/drive/MyDrive/YOLO_Project'
# RUNS_DET_DIR = os.path.join(PROJECT_DIR, 'runs_det')

# # 기존 학습 결과 폴더 초기화
# if os.path.exists(RUNS_DET_DIR):
#     shutil.rmtree(RUNS_DET_DIR)
#     print("🧹 기존 runs_det 결과 폴더 삭제 완료!")
# else:
#     print("📂 삭제할 runs_det 폴더가 없습니다. 깔끔한 상태입니다!")

# 이 셀 또한 위와같은 오류가 발생할 경우, 최초 1회 실행 후 모두 주석 처리

In [17]:
# yolov12 가중치 수동 다운로드
import os
import urllib.request

PROJECT_DIR = '/content/drive/MyDrive/YOLO_Project'
v12_weight_path = os.path.join(PROJECT_DIR, "yolo12n.pt")  # 파일명에서 v 제거 (yolo12n.pt)

# YOLOv12 가중치 수동 다운로드 (수정된 URL & 파일명)
if not os.path.exists(v12_weight_path):
    print("📥 YOLOv12 가중치 다운로드 중...")
    try:
        url = "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo12n.pt"
        urllib.request.urlretrieve(url, v12_weight_path)
        print("✅ YOLOv12 가중치 다운로드 완료!")
    except Exception as e:
        print(f"⚠️ YOLOv12 가중치 다운로드 실패: {e}")
else:
    print("✅ 이미 YOLOv12 가중치 파일이 존재합니다.")

✅ 이미 YOLOv12 가중치 파일이 존재합니다.


In [15]:
# 모델 순차 학습, 성능 비교표 출력 및 최적 가중치(best.pt) 다운로드

import os
import pandas as pd
from ultralytics import YOLO
from google.colab import files

PROJECT_DIR = '/content/drive/MyDrive/YOLO_Project'
RUNS_DET_DIR = os.path.join(PROJECT_DIR, 'runs_det')
YAML_PATH = os.path.join(PROJECT_DIR, 'dataset_det', 'data.yaml')

v10_weight_path = os.path.join(PROJECT_DIR, "yolov10n.pt")
v12_weight_path = os.path.join(PROJECT_DIR, "yolo12n.pt")

# 1. YOLOv10 학습 및 검증
print("🚀 [1/2] YOLOv10 Detection 학습 시작")
model_v10 = YOLO(v10_weight_path)
model_v10.train(
    data=YAML_PATH,
    epochs=30,
    imgsz=640,
    batch=16,
    project=RUNS_DET_DIR,
    name='v10_experiment',
    exist_ok=True
)
metrics_v10 = model_v10.val()

# 2. YOLOv12 학습 및 검증
print("\n🚀 [2/2] YOLOv12 Detection 학습 시작")
model_v12 = YOLO(v12_weight_path if os.path.exists(v12_weight_path) else 'yolo12n.pt')
model_v12.train(
    data=YAML_PATH,
    epochs=30,
    imgsz=640,
    batch=16,
    project=RUNS_DET_DIR,
    name='v12_experiment',
    exist_ok=True
)
metrics_v12 = model_v12.val()

# 3. 성능 비교 리포트 출력
print("\n" + "="*50)
print("📊 YOLOv10 vs YOLOv12 Detection 성능 비교 리포트")
print("="*50)

df_compare = pd.DataFrame({
    'Model': ['YOLOv10n', 'YOLOv12n'],
    'Precision (정밀도)': [metrics_v10.box.mp, metrics_v12.box.mp],
    'Recall (재현율)': [metrics_v10.box.mr, metrics_v12.box.mr],
    'mAP50': [metrics_v10.box.map50, metrics_v12.box.map50],
    'mAP50-95': [metrics_v10.box.map, metrics_v12.box.map]
})

print(df_compare.to_string(index=False))
print("-" * 50)

best_model_name = "YOLOv10" if metrics_v10.box.map50 >= metrics_v12.box.map50 else "YOLOv12"
print(f"🏆 mAP50 기준 최종 우수 모델: {best_model_name}")

# 4. 학습 완료 후 가중치 파일(best.pt) 자동 다운로드
v10_best_path = os.path.join(RUNS_DET_DIR, 'v10_experiment', 'weights', 'best.pt')
v12_best_path = os.path.join(RUNS_DET_DIR, 'v12_experiment', 'weights', 'best.pt')

print("\n💾 최적 가중치 파일(best.pt) 다운로드를 시작합니다...")
if os.path.exists(v10_best_path):
    files.download(v10_best_path)
if os.path.exists(v12_best_path):
    files.download(v12_best_path)

🚀 [1/2] YOLOv10 Detection 학습 시작
Ultralytics 8.4.170 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/YOLO_Project/dataset_det/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/YOLO_Project/yolov10n.pt, momentum=0.937

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 성능 높이기  
##### 현재 모델이 더 개선될 여지가 있는 핵심 지표는 mAP50-95  
##### (mAP50-95가 0.458 정도 -> 더 끌어올릴 수 있음)  
- mAP50-95는 바운딩 박스의 경계선 위치를 얼마나 정밀하게(IoU 50% ~ 95% 구간) 맞추는지 평가하는 지표

#### imgsz (입력 이미지 크기)  
##### (위 코드의 imgsz는 640)  
- 만약 탐지하려는 객체가 이미지 내에서 작거나, 경계선을 더 정밀하게 잡고 싶다면  
  *   imgsz=1024 또는 imgsz=1280으로 올리는 것이 성능 향상에 크게 기여
  (Gemini는 1024 추천)



#### batch (배치 사이즈)  
##### (위 코드의 batch는 16)  
- 영향: 배치 사이즈가 크면 학습 안정성이 높아지고, 작으면 약간의 노이즈가 섞여 미니마를 벗어나는 데 도움이 될 수도 있습니다.
  *   T4 GPU 환경에서는 batch=16 또는 batch=32가 가장 안정적


#### epochs (학습 횟수)  
##### (위 코드의 epochs는 30)  
- mAP50-95 정밀도 학습에는 50~100 epochs 정도의 충분한 학습이 필요  
- 특히 뒤쪽 epoch에서 모자이크 증강(Mosaic augmentaion)이 꺼지면서 (close_mosaic=10) 경계선 세부 학습이 집중적으로 이루어지므로, 50 epoch 이상으로 늘리는 것이 효과적

### 성능(mAP50-95)을 추가로 끌어올리기 위한 3가지 전략  
- 입력 이미지 크기 확대 (imgsz=1024)
  *   이미지 해상도를 키우면 객체의 미세한 테두리 정보를 훨씬 잘 포착하므로 mAP50-95 지표 향상에 가장 직접적인 효과가 있습니다  
- Epoch 수 증가 (epochs=50 또는 100)
  * 바운딩 박스 위치를 맞추는 DFL(Distribution Focal Loss) 및 Box Loss가 충분히 줄어들 수 있도록 학습 시간을 더 줍니다  
- 더 큰 모델 사이즈 사용 (yolov10m.pt 또는 yolov10l.pt)
  * 현재 사용하신 모델은 가장 가볍고 빠른 n (nano) 버전입니다. 모델 파라미터 수가 많은 m (medium)이나 l (large) 버전을 사용하면 복잡한 특징 표현력이 높아져 성능이 크게 향상됩니다.

